# Demo (Modular, Deterministic, No Normalizer)

Minimal, testable steps:
1) Create/verify directories + PYTHONPATH
2) Clone phenopacket-store, build PMID list, fetch PDFs, build dataset CSV
3) Load dataset, validate columns, dedupe
4) **PDF → Text verification (preview + full dumps to disk)** ← _inspect conversion here_
5) Align ground-truth phenopackets
6) Sanity inference (single case) — LLM must infer exact HPO `id` + primary `label` itself
7) Batch inference
8) Evaluation

**Important:** There is **no** post-hoc HPO normalization or ontology lookup. The model must produce the exact `HP:#######` id and the official primary label by itself. We use `temperature=0` and structured JSON output to reduce drift.

In [30]:
# --- Step 1: Directory creation + PYTHONPATH setup ---

import sys, os

# Ensure project root (this notebook lives under notebooks/)
project_root_from_notebook = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root_from_notebook not in sys.path:
    sys.path.insert(0, project_root_from_notebook)

from notebooks.utils.demonstration_directory_creation import (
    patch_pythonpath_and_create_demonstration_directories,
)

paths = patch_pythonpath_and_create_demonstration_directories(emit_verbose_logs=True)

# Re-export with ultra-clear names
project_root = str(paths.project_root_directory)
src_folder = str(paths.source_code_directory)
utils_folder = str(paths.notebooks_utilities_directory)
pdf_input_directory = str(paths.pdf_input_directory)
ground_truth_notebooks_directory = str(paths.ground_truth_notebooks_directory)
dataset_csv_path = str(paths.dataset_csv_file_path)
experimental_data_root = str(paths.experimental_data_root_directory)
llm_output_directory = str(paths.llm_raw_output_directory)
validated_jsons_directory = str(paths.validated_jsons_output_directory)
evaluation_report_output_path = str(paths.evaluation_report_file_path)

print("[OK] Step 1 complete: directories + PYTHONPATH are set up.")


PYTHONPATH patched with: /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/notebooks/utils
Project Root:         /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5
Source Folder:        /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src
Utilities Folder:     /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/notebooks/utils
Created/checked PDF input folder:                  /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs
Created/checked ground-truth notebooks folder:     /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laborato

In [31]:
# --- Step 2: Phenopacket-store dataset bootstrap (Stages 1–4) ---
# Requires Step 1 vars:
#   - src_folder, pdf_input_directory, ground_truth_notebooks_directory, dataset_csv_path

from notebooks.utils.phenopacket_store_dataset_setup import setup_phenopacket_store_dataset

# Choose how many PDFs to download:
#   0 = all available; N = first N (useful for quick smoke tests)
maximum_pdf_download_count_for_setup = 10

pmid_pickle_file_path = setup_phenopacket_store_dataset(
    src_folder=src_folder,
    pdf_input_directory=pdf_input_directory,
    ground_truth_notebooks_directory=ground_truth_notebooks_directory,
    dataset_csv_path=dataset_csv_path,
    max_pdfs_to_download=maximum_pdf_download_count_for_setup,
)

print("[OK] Step 2 complete: phenopacket-store bootstrapped.")
print("pmid_pickle_file_path:", pmid_pickle_file_path)
print("dataset_csv_path:", dataset_csv_path)


[Stage 0] Preparing ground-truth notebooks directory for a fresh clone...
  - Removing existing directory: /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/notebooks
[Stage 1] Cloning 'phenopacket-store' notebooks...
  - Stage 1 Complete. Produced: /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/notebooks
[Stage 2] Scanning notebooks for PMIDs and creating pmids.pkl...
1237 PMIDs found within /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/notebooks
  - Stage 2 Complete.
[Stage 3] Downloading PDFs for discovered PMIDs...


Processing PMID_33855675:  10%|█▋               | 1/10 [00:07<01:04,  7.15s/it]

A PDF for PMID_33855675 was successfully downloaded. PMCID=11002654.


Processing PMID_29078790:  20%|███▍             | 2/10 [00:13<00:54,  6.77s/it]

A PDF for PMID_29078790 was successfully downloaded. PMCID=5658948.


...

No PMCID found for PMID_30558828.


Processing PMID_11175294:  90%|███████████████▎ | 9/10 [00:40<00:03,  3.50s/it]

No PMCID found for PMID_11175294.
  - Stage 3 Complete.
[Stage 4] Building dataset CSV (if missing)...


Processing of 10 PMIDs complete. 5 PDFs successfully downloaded.: 100%|█| 10/10


  - Created dataset CSV at: /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/PMID_PDF_Phenopacket_list_in_phenopacket_store.csv
  - Stage 4 Complete.
Summary of created/verified paths:
  - PDF inputs folder:        /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs
  - Ground truth folder:      /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/notebooks
  - Dataset CSV path:         /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/PMID_PDF_Phenopacket_list_in_phenopacket_store.csv
[OK] Step 2 complete: phenopacket-store bootstrapped.
pmid_pickle_file_path: /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Geno

In [32]:
# --- Step 3: Load dataset CSV, validate columns, dedupe by PMID, quick preview ---

from notebooks.utils.dataset_loading import load_and_validate_dataset
from IPython.display import display

# How many “input” paths to quickly existence-check (set 0 to skip)
max_input_existence_checks = 10

dataframe_cases, dataset_stats = load_and_validate_dataset(
    dataset_csv_path=dataset_csv_path,
    max_input_existence_checks=max_input_existence_checks,
    verbose=True,
)

print("\n[dataset] Stats:", dataset_stats)

print("\n[dataset] Preview (first 5 rows):")
display(dataframe_cases.head(5))

print("hello1")  # sanity check


[dataset] Loaded 167 row(s) from CSV: /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/PMID_PDF_Phenopacket_list_in_phenopacket_store.csv
[dataset] Checking existence of first 10 input files:
  - /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs/PMID_29290338.pdf: FOUND
  - /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs/PMID_29290338.pdf: FOUND
  - /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs/PMID_29290338.pdf: FOUND
  - /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs/PMID_29290338.

,pmid,input,truth
0,PMID_29290338,/Users/varenya/Desktop/Illini+Uni/Personalized...,/Users/varenya/Desktop/Illini+Uni/Personalized...
1,PMID_29078790,/Users/varenya/Desktop/Illini+Uni/Personalized...,/Users/varenya/Desktop/Illini+Uni/Personalized...
2,PMID_33855675,/Users/varenya/Desktop/Illini+Uni/Personalized...,/Users/varenya/Desktop/Illini+Uni/Personalized...
3,PMID_20799361,/Users/varenya/Desktop/Illini+Uni/Personalized...,/Users/varenya/Desktop/Illini+Uni/Personalized...
4,PMID_26453364,/Users/varenya/Desktop/Illini+Uni/Personalized...,/Users/varenya/Desktop/Illini+Uni/Personalized...


hello1


In [33]:
# --- Step 4: Initialize PDF→text cache, preview & dump full text (for verification) ---

import os
from notebooks.utils.pdf_text_cache import PdfTextCache

pdf_text_cache = PdfTextCache(experimental_data_root=experimental_data_root)

# Convert a few inputs to verify pipeline correctness
#num_smoke_tests = min(3, len(dataframe_cases))
num_smoke_tests = len(dataframe_cases)
print(f"[text] Converting first {num_smoke_tests} document(s) to text...")

# Where to write full text dumps for inspection
debug_dump_directory = os.path.join(experimental_data_root, "text_cache", "debug_dumps")
os.makedirs(debug_dump_directory, exist_ok=True)

for index_counter, input_file_path in enumerate(dataframe_cases["input"].head(num_smoke_tests), start=1):
    try:
        extracted_text = pdf_text_cache.get_text(input_file_path)

        # Human-readable preview (whitespace collapsed)
        collapsed = " ".join(extracted_text.split())
        preview_length = 1200  # characters
        print(f"  [{index_counter}] {input_file_path} → {len(extracted_text)} characters")
        print(f"      preview (first {preview_length} chars): {collapsed[:preview_length]}...")

        # Save the full text for inspection in an editor
        base_name = os.path.splitext(os.path.basename(input_file_path))[0]
        debug_text_path = os.path.join(debug_dump_directory, base_name + ".txt")
        with open(debug_text_path, "w", encoding="utf-8") as debug_handle:
            debug_handle.write(extracted_text)
        print(f"      saved full text → {debug_text_path}")

    except Exception as e:
        print(f"  [{index_counter}] {input_file_path} → ERROR: {e}")

print("hello2")  # sanity check


[text] Converting first 5 document(s) to text...


/Users/varenya/miniconda3/envs/P5/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Parameter `strict_text` has been deprecated and will be ignored.


  [1] /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs/PMID_29290338.pdf → 150087 characters
      preview (first 1200 chars): ## Genotype-Phenotype Correlation in NF1: Evidence for a More Severe Phenotype Associated with Missense Mutations Affecting NF1 Codons 844-848 Magdalena Koczkowska, 1 Yunjia Chen, 1 Tom Callens, 1 Alicia Gomes, 1 Angela Sharp, 1 Sherrell Johnson, 1 Meng-Chang Hsiao, 1 Zhenbin Chen, 1 Meena Balasubramanian, 2 Christopher P. Barnett, 3 Troy A. Becker, 4 Shay Ben-Shachar, 5 Debora R. Bertola, 6 Jaishri O. Blakeley, 7 Emma M.M. Burkitt-Wright, 8 Alison Callaway, 9 Melissa Crenshaw, 4 Karin S. Cunha, 10 Mitch Cunningham, 11 Maria D. D'Agostino, 12 Karin Dahan, 13 Alessandro De Luca, 14 Anne Destre ´e, 13 Radhika Dhamija, 15 Marica Eoli, 16 D. Gareth R. Evans, 8 Patricia Galvin-Parton, 17 Jaya K. George-Abraham, 18 Karen W. Gripp, 19 Jose Guevara-Campos, 20 Neil

/Users/varenya/miniconda3/envs/P5/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Parameter `strict_text` has been deprecated and will be ignored.


  [2] /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs/PMID_29078790.pdf → 23842 characters
      preview (first 1200 chars): ## CASE REPORT ## Hereditary neuropathy with liability to pressure palsy (HNPP): report of a family with a new point mutation in PMP22 gene Carlo Fusco 1,2 , Carlotta Spagnoli 1* , Grazia Gabriella Salerno 1 , Elena Pavlidis 1 , Daniele Frattini 1 and Francesco Pisani 3 ## Abstract Background: Hereditary neuropathy with liability to pressure palsy (HNPP) is an autosomal dominant disorder most commonly presenting with acute-onset, non-painful focal sensory and motor mononeuropathy. Approximately 80% of patients carry a 1.5 Mb deletion of chromosome 17p11.2 involving the peripheral myelin protein 22 gene (PMP22), the same duplicated in Charcot-Marie-Tooth 1A patients. In a small proportion of patients the disease is caused by PMP22 point mutations. Case prese

...

  [3] /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs/PMID_33855675.pdf → 37794 characters
      preview (first 1200 chars): Published in final edited form as: J Clin Immunol. 2021 August ; 41(6): 1241-1249. doi:10.1007/s10875-021-01035-1. ## Abnormal SCID Newborn Screening and Spontaneous Recovery Associated with a Novel Haploinsufficiency IKZF1 Mutation Hye Sun Kuehn 1 , Nicholas J. Gloude 2,3 , David Dimmock 4 , Mari Tokita 4 , Meredith Wright 4 , Sergio D. Rosenzweig 1 , Cathleen Collins 3,5 - 1 Immunology Service, Department of Laboratory Medicine, NIH Clinical Center, National Institutes of Health, Building 10, Rm 2C306, 10 Center Drive, MSC1508, Bethesda, MD, USA - 2 Division of Hematology Oncology, Department of Pediatrics, University of California San Diego, San Diego, CA, USA - 3 Rady Children's Hospital San Diego, San Diego, CA, USA - 4 Rady Children's Institute for Ge

/Users/varenya/miniconda3/envs/P5/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/varenya/miniconda3/envs/P5/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Parameter `strict_text` has been deprecated and will be ignored.


  [4] /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs/PMID_20799361.pdf → 24450 characters
      preview (first 1200 chars): Published in final edited form as: Am J Hematol. 2010 October ; 85(10): 824-828. doi:10.1002/ajh.21836. ## Hemolytic anemia and distal renal tubular acidosis in two Indian patients homozygous for SLC4A1/AE1 mutation A858D Boris E. Shmukler *,1,2 , Prabhakar S. Kedar *,1,2,3 , Prashant Warang 3 , Mukesh Desai 4 , Manisha Madkaikar 3 , Kanjaksha Ghosh 3 , Roshan B. Colah 3 , and Seth L. Alper 1,2 1 Renal Division and Molecular and Vascular Medicine Unit, Beth Israel Deaconess Medical Center 2 Department of Medicine, Harvard Medical School, Boston, MA 3 National Institute of Immunohematology, Indian Council of Medical Research 4 Division of Immunology and Dept. of Hematology Oncology, B.J. Wadia Children's Hospital, Mumbai, India ## Abstract Familial distal re

Parameter `strict_text` has been deprecated and will be ignored.


  [5] /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs/PMID_26453364.pdf → 19309 characters
      preview (first 1200 chars): ## RESEARCH REPORT ## Further Delineation of the ALG9-CDG Phenotype Sarah AlSubhi • Amal AlHashem • Anas AlAzami • Kalthoum Tlili • Saad AlShahwan • Dirk Lefeber • Fowzan S. Alkuraya • Brahim Tabarki Received: 13 July 2015 /Revised: 14 September 2015 /Accepted: 18 September 2015 /Published online: 10 October 2015 # SSIEM and Springer-Verlag Berlin Heidelberg 2015 Abstract ALG9-CDG is one of the less frequently reported types of CDG. Here, we summarize the features of six patients with ALG9-CDG reported in the literature and report the features of four additional patients. The patients presented with drug-resistant infantile epilepsy, hypotonia, dysmorphic features, failure to thrive, global developmental disability, and skeletal dysplasia. One patient prese

In [34]:
# --- Step 5: Load & align ground-truth phenopackets with the dataset ---

from notebooks.utils.truth_alignment import load_and_align_truth

# Set to a small integer for quick smoke tests, or None to process all rows
maximum_rows_to_align = None

truth_alignment = load_and_align_truth(
    dataset_dataframe=dataframe_cases,
    maximum_rows=maximum_rows_to_align,
)

pmids_aligned = truth_alignment.list_of_pmids_aligned
truth_packets_wrapped = truth_alignment.list_of_truth_packets_wrapped
patient_ids_from_truth = truth_alignment.list_of_patient_ids_from_truth
input_paths_aligned = truth_alignment.list_of_input_paths_aligned
skipped_truth_cases = truth_alignment.list_of_skipped_cases

print(f"[truth] aligned cases: {len(pmids_aligned)}")
if skipped_truth_cases:
    print(f"[truth] skipped cases: {len(skipped_truth_cases)} (showing up to 5)")
    for item in skipped_truth_cases[:5]:
        print("   -", item)

# Optional: quick peek at first packet’s phenotypes
if truth_packets_wrapped:
    try:
        first_pheno_preview = truth_packets_wrapped[0].list_phenotypes()
        print(f"[truth] first packet phenotype count: {len(first_pheno_preview)}")
    except Exception as e:
        print(f"[truth] preview error: {e}")

print("hello3")  # sanity check


[truth] aligned cases: 5
[truth] first packet phenotype count: 12
hello3


In [35]:
# --- Step 6: Sanity inference with the LLM (first aligned case) ---
import os, json
from notebooks.utils.hpo_extraction import (
    extract_hpo_terms,
    build_minimal_phenopacket_from_hpo_list,
)

# Reuse the text conversion via the cache to guarantee parity with batch
from notebooks.utils.pdf_text_cache import PdfTextCache
pdf_text_cache = PdfTextCache(experimental_data_root=experimental_data_root)

if not pmids_aligned:
    raise RuntimeError("No aligned cases available. Ensure the truth-alignment step succeeded.")

example_index = 0  # choose the first aligned case for a quick smoke test
example_pmid = pmids_aligned[example_index]
example_patient_id = patient_ids_from_truth[example_index]
example_input_path = input_paths_aligned[example_index]

print(f"[inference] Using PMID={example_pmid} | patient_id={example_patient_id}")
print(f"[inference] Source file: {example_input_path}")

# Load clinical text from cache (same as Stage 4 paths)
clinical_text_for_example = pdf_text_cache.get_text(example_input_path)
print(f"[inference] Loaded {len(clinical_text_for_example)} characters of clinical text")

# Run the extractor (model must reason exact HP id + label; deterministic JSON)
predicted_terms_for_example, raw_model_text = extract_hpo_terms(
    clinical_text=clinical_text_for_example,
    model="llama3.2:latest",   # adjust if needed
    return_raw_model_text=True,
    debug_logging=True,
    max_pheno_items=20,         # soft cap (we still drop invalids)
)

print(f"[inference] extracted {len(predicted_terms_for_example)} HPO term object(s)")
print(json.dumps(predicted_terms_for_example[:5], indent=2))  # preview first few

# Wrap into a minimal Phenopacket (no normalization)
from phenopacket import Phenopacket as UtilPhenopacket

predicted_packet_json = build_minimal_phenopacket_from_hpo_list(
    patient_id=example_patient_id,
    hpo_list=predicted_terms_for_example,
)
predicted_packet_util = UtilPhenopacket(predicted_packet_json)
print("[inference] phenotypicFeatures count:", len(predicted_packet_util.list_phenotypes()))

# Persist raw model text for auditing
os.makedirs(llm_output_directory, exist_ok=True)
raw_dump_path = os.path.join(llm_output_directory, f"{example_pmid}__raw.txt")
with open(raw_dump_path, "w", encoding="utf-8") as raw_handle:
    raw_handle.write(raw_model_text or "")
print(f"[inference] saved raw model output → {raw_dump_path}")

print("hello4")  # sanity check


[inference] Using PMID=PMID_29290338 | patient_id=Family UAB-R45201FN.101 individual RS
[inference] Source file: /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/src/P5/scripts/data/tmp/phenopacket_store/pmid_pdfs/PMID_29290338.pdf
[inference] Loaded 150087 characters of clinical text
[extract] Sending prompt to model with deterministic settings (temperature=0).
[extract] Raw model text length: 2
[extract] Model did not return a list; dropping output.
[extract] Validated 0 HPO term(s).
[inference] extracted 0 HPO term object(s)
[]
[inference] phenotypicFeatures count: 0
[inference] saved raw model output → /Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/experimental-data/llm_output_dir/PMID_29290338__raw.txt
hello4


In [36]:
# --- Step 7: Batch inference over all aligned cases ---

from notebooks.utils.hpo_batch import run_hpo_batch_inference

# Pull aligned lists from the earlier truth-alignment object
aligned_pmids = truth_alignment.list_of_pmids_aligned
aligned_input_paths = truth_alignment.list_of_input_paths_aligned
aligned_patient_ids = truth_alignment.list_of_patient_ids_from_truth

print(f"[batch] total aligned cases: {len(aligned_pmids)}")

batch_result = run_hpo_batch_inference(
    list_of_pmids_aligned=aligned_pmids,
    list_of_input_paths_aligned=aligned_input_paths,
    list_of_patient_ids_aligned=aligned_patient_ids,
    directory_for_raw_llm_outputs=llm_output_directory,
    directory_for_predicted_jsons=validated_jsons_directory,
    ollama_model_name="llama3.2:latest",   # adjust model here if desired
    sleep_seconds_between_cases=0.0,        # add delay if you want rate limiting
    write_raw_model_text=True,
    write_predicted_json=True,
    debug_logging=False,                    # set True for verbose logging
)

print(f"[batch] successes = {batch_result.total_successes()} | failures = {batch_result.total_failures()}")

# Keep the predicted packet utils handy for evaluation
predicted_packet_utils_for_evaluation = [o.predicted_packet_util for o in batch_result.successful_outcomes]

# Quick peek at a couple of outputs (paths)
for outcome in batch_result.successful_outcomes[:3]:
    print(f"  ✓ PMID {outcome.pmid}: raw={outcome.raw_output_path} | json={outcome.predicted_json_path}")

if batch_result.failed_outcomes:
    print("\n[batch] Failures (first 5 shown):")
    for outcome in batch_result.failed_outcomes[:5]:
        print(f"  ✗ PMID {outcome.pmid}: {outcome.error_message}")

print("hello5")  # sanity check


[batch] total aligned cases: 5


/Users/varenya/miniconda3/envs/P5/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Parameter `strict_text` has been deprecated and will be ignored.
/Users/varenya/miniconda3/envs/P5/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Parameter `strict_text` has been deprecated and will be ignored.
/Users/varenya/miniconda3/envs/P5/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/varenya/miniconda3/envs/P5/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argumen

[batch] successes = 5 | failures = 0
  ✓ PMID PMID_29290338: raw=/Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/experimental-data/llm_output_dir/PMID_29290338__raw.txt | json=/Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/experimental-data/validated_jsons/PMID_29290338__Family UAB-R45201FN.101 individual RS.json
  ✓ PMID PMID_29078790: raw=/Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/experimental-data/llm_output_dir/PMID_29078790__raw.txt | json=/Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/experimental-data/validated_jsons/PMID_29078790__proband.json
  ✓ PMID PMID_33855675: raw=/Users/varenya/Desktop/Illini+Uni/Personalized_Genomic_Medicine-Precision_Genomics_Laboratory/PreGen/P5/experimental-data/llm_output_dir/PMID_33855675__raw.txt | json=/Users/varenya/D

In [37]:
# --- Step 8: Evaluate predictions vs. ground truth ---

from evaluation import PhenotypeEvaluator
from report import Report

if not predicted_packet_utils_for_evaluation:
    raise RuntimeError("No predicted packets available for evaluation. Run the batch step first.")

# Pair predictions with truth 1:1 on the successful subset indices
predicted_by_pmid = {
    outcome.pmid: outcome.predicted_packet_util
    for outcome in (getattr(globals().get('batch_result', None), 'successful_outcomes', []) or [])
    if outcome.predicted_packet_util is not None
}

truth_packets_for_eval = []
predicted_packets_for_eval = []
for pmid, truth_pp in zip(aligned_pmids, truth_alignment.list_of_truth_packets_wrapped):
    if pmid in predicted_by_pmid:
        truth_packets_for_eval.append(truth_pp)
        predicted_packets_for_eval.append(predicted_by_pmid[pmid])

print(f"[eval] evaluating {len(predicted_packets_for_eval)} predicted packets")

evaluator = PhenotypeEvaluator()
for predicted_pp, truth_pp in zip(predicted_packets_for_eval, truth_packets_for_eval):
    evaluator.check_phenotypes(predicted_pp.list_phenotypes(), truth_pp)

final_report = evaluator.report(
    creator="P5-demo-notebook",
    experiment="LLM HPO Extraction (HPO-only eval vs phenopacket-store)",
    model="ollama:llama3.2:latest",
    notes=f"Total evaluated pairs: {len(predicted_packets_for_eval)}",
)

try:
    print("=== Evaluation Summary ===")
    print(final_report.get_summary())
except Exception:
    print("=== Evaluation Summary (raw) ===")
    print(getattr(final_report, "metrics", "<no metrics>"))
    print(getattr(final_report, "metadata", "<no metadata>"))

print("hello6")  # sanity check


[eval] evaluating 5 predicted packets
=== Evaluation Summary ===
=== Evaluation Summary (raw) ===
{'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0}
{'creator': 'P5-demo-notebook', 'experiment': 'LLM HPO Extraction (HPO-only eval vs phenopacket-store)', 'model': 'ollama:llama3.2:latest', 'date': '2025-08-29', 'notes': 'Total evaluated pairs: 5'}
hello6
